In [1]:
#README: This file is to test out making animations in Makie and optimizing the performance. 
using Pkg 
Pkg.activate("cell_migration_julia_v1.10.0"; shared = false)
Pkg.project().path

  Activating new project at `C:\Users\mbarm\Documents\Documents\Code\cell_migration_julia_v1.10.0`


"C:\\Users\\mbarm\\Documents\\Documents\\Code\\cell_migration_julia_v1.10.0\\Project.toml"

In [25]:
using Base64
using Markdown
using NBInclude
using Base: match
using LaTeXStrings
using OrderedCollections

using Base64
using UUIDs

#Random Numbers & Statistics 
using Random
using StatsBase
using Distributions

#Plotting, gifs, MP4s 
#using Plots
using CairoMakie
using Colors
using Measures  
using FFMPEG

#File IO 
using CSV
using DelimitedFiles

#Printing
using Printf
using LaTeXStrings

#Data Analysis 
using DataFrames

#Numerical Methods & related tools
using DifferentialEquations
using ForwardDiff
using LinearAlgebra
using StaticArrays
using PreallocationTools
using FastGaussQuadrature

#Benchmarking & Progress 
using BenchmarkTools
using ProgressMeter
ProgressMeter.ijulia_behavior(:clear);

#Misc 
using Dates
using UnPack
using Logging
using NBInclude
using Base.Threads
using OrderedCollections

#Import some notebooks 
@nbinclude("C:/Users/mbarm/Documents/Documents/Code/Research/Cell Models/Migration/ShreyasModel2.ipynb")
@nbinclude("C:/Users/mbarm/Documents/Documents/Code/Research/Cell Models/Migration/OrderParameters.ipynb")
@nbinclude("C:/Users/mbarm/Documents/Documents/Code/Research/Cell Models/Migration/ShreyasHelpers.ipynb")

#Set the directory for saving output files (plots, mp4s, txt files, etc.) 
output_dir = "C:\\Users\\mbarm\\Documents\\Documents\\Research\\Cell Models\\ShreyasModel\\"

#Set the current directory to be the code directory 
cd(output_dir)   

#Check that the current directory was set correctly
pwd()              

  Activating project at `C:\Users\mbarm\.julia\environments\cell_migration_julia_v1.10.0`
  Activating project at `C:\Users\mbarm\.julia\environments\cell_migration_julia_v1.10.0`
  Activating project at `C:\Users\mbarm\.julia\environments\cell_migration_julia_v1.10.0`
  Activating project at `C:\Users\mbarm\.julia\environments\cell_migration_julia_v1.10.0`


"C:\\Users\\mbarm\\Documents\\Documents\\Research\\Cell Models\\ShreyasModel"

In [26]:
#SET MAKIE'S DEFAULT FONT THEME TO 'COMPUTER ROMAN' [https://discourse.julialang.org/t/makie-jl-latex-ticks-and-other-elements-by-default/102529/3]
#Once this is added in the environment, do we really need to run this every time? 
MT = Makie.MathTeXEngine
mt_fonts_dir = joinpath(dirname(pathof(MT)), "..", "assets", "fonts", "NewComputerModern")
set_theme!(fonts = (regular = joinpath(mt_fonts_dir, "NewCM10-Regular.otf"), bold = joinpath(mt_fonts_dir, "NewCM10-Bold.otf"),
                    italic = joinpath(mt_fonts_dir, "NewCM10-Italic.otf")))

In [27]:
#MAKIE VERSION OF ANIMATION FUNCTION and DISPLAY VIDEO FUNCTION 
#ATTEMPT AT OPTIMIZING THE PERFORMACNE OF animated_scatter_with_quiver!()


function animated_scatter_with_quiver(df::DataFrame; cols_scatter::Vector{Symbol}, cols_quiver::Vector{Symbol}, fps = 30,
                                      xlim = (-500,500), ylim = (-500,500), temperature::Function = cue, arrowlength = 50.0, 
                                      title::AbstractString = "", size = (600, 400), index::Vector = collect(0:nrow(df)-1),
                                      particle_diameter::Real = 20.0)
    
    """
    MAKIE VERSION OF ANIMATION FUNCTION and DISPLAY VIDEO FUNCTION 
    
    Parameters
    ----------
    df :: A time series stored in a DataFrame, where the rows are the data at different times and the columns are the different variables.  
    cols_scatter :: The names of the columns in df containing the points (Vector{Float64}'s or Tuple{Float64, Float64}'s) to be scattered. 
    cols_quiver :: The names of the columns in df containing the points/vectors for drawing arrows. 
    fps :: The frame rate (frames per second)
    xlim, ylim :: The x- and y-axis limits for the scatter plots. 
    temperature :: The function for drawing the heatmap background of the scatter plots. 
    arrowlength :: The length of the arrows to be drawn. 
    title :: The title for each of the frames. (This part is static, but we also add a dynamic title to indicate the current time value.) 
    size :: The tuple in the form (width, height) giving the dimensions of the frames in pixels. 
    index :: The time values by which the data in df is indexed. 
    particle_diameter :: The width of the particles (in whichever units xlim and ylim have). 
    
    Returns
    -------
    video_bytes::Vector{UInt8}
                 The plot data that is used to create an mp4 file. 
    """
    
    ########################
    #SET UP A FIGURE & AXIS
    ########################
    
    fig = Figure(size = size)
    grid = fig[1, 1] = GridLayout()
    xmin, xmax = xlim; ymin, ymax = ylim     
    ax = Axis(fig[1, 1], title = title, xlabel = L"x", ylabel = L"y", xlabelsize = 14, ylabelsize = 14,
              titlesize = 14, titlealign = :left, limits = (xmin, xmax, ymin, ymax), ylabelrotation = 0)

    #######################
    #COMPUTE THE MARKERSIZE 
    #######################

    #Here we compute the markersize for the cells so that they are drawn to scale.
    #WANT: (particle diameter / domain length) = (diameter of marker [in pixels]) / (with of plot [in pixels])
    domain_length = xmax - xmin 
    axis_width_in_pixels = CairoMakie.width(ax.layoutobservables.computedbbox[])             #NOTE: this value is unaffected by xlim and yxlim 
    markerwidth = (particle_diameter / domain_length) * axis_width_in_pixels

    ########
    #COLORS#
    ########

    marker_color = RGBA(118/255, 96/255, 205/255, 1.0)    #note alpha is 0 to 1, so 255/255 = 1.0
    arrow_color = RGBA(0.9, 0.9, 0.98, 0.5)               #Lavender with 80% opacity 
    heatmap_colors = cgrad(["#fff5ec", "#fde0c9", "#fac8ac", "#f5a98c", "#eb7962", "#d45d4c", "#bb4a40"])
    
    #############################
    #CREATE A HEATMAP & COLORBAR
    #############################
    
    #Create a static heatmap 
    xgrid = LinRange(xmin, xmax, 200)
    ygrid = LinRange(ymin, ymax, 2)
    heat_vals = [cue([x,y]) for x in xgrid, y in ygrid]
    hm = heatmap!(ax, xgrid, ygrid, heat_vals, colormap = heatmap_colors)

    #Create a colorbar for the heatmap 
    cb = Colorbar(fig[1,2], hm; label = "cue", width = 12)
    
    #Reduce whitespace between plot and colorbar. See this: https://docs.makie.org/stable/tutorials/aspect-tutorial.html
    colsize!(fig.layout, 1, Aspect(1, 1.0))  
    
    ################################
    #SET UP SCATTER PLOT WITH ARROWS
    ################################

    num_frames = nrow(df)
    num_points = length(cols_scatter)     #number of points in each frame (= the number of cells) 
  
    vars_scatter = df[:, cols_scatter]    #subset the columns containing the positions (for the scatter plot)
    vars_quiver = df[:, cols_quiver]      #subset the columns containing the arrows (for the quiver) 
    
    x0 = vars_scatter[1, :]               #The initial positions
    w0 = vars_quiver[1, :]                #The initial polarities 

    points = Observable([Point2f(x) for x in x0])      #Convert the initial positions to Point2f0's 
    vecs = Observable([Point2f(w) for w in w0])        #Convert the initial polarities to Point2f0's 
   
    scatter!(ax, points, color = marker_color, markersize = markerwidth)
    arrows2d!(ax, points, vecs, color = arrow_color, normalize = true, lengthscale = arrowlength, tipwidth = 6, shaftwidth = 1, tiplength = 4)
    #NOTE: normalize = true ==> the arrow length = lengthscale 

    #######################################
    #ADD A LABEL FOR THE CURRENT TIME VALUE 
    ########################################

    #NOTE: label must be created AFTER creating the heatmap; otherwise the label will be hidden.  
    t_str =  @sprintf("%5d", index[1]) 
    t_label = Observable(L"t = %$(t_str)")
    text!(fig.scene, Point2f(0.7, 0.968), text = t_label, space = :relative, align = (:left, :top))
    
    #####################
    #CREATE AN ANIMATION
    #####################

    # Save to a unique temp file
    filename = tempname() * ".mp4"
    record(fig, filename, 1:num_frames; framerate = fps) do i

        #FIGURE OUT HOW TO REDUCE ALLOCATIONS HERE!!! MAYBE CONVERT THE DF TO AN ARRAY OF Points/SVectors BEFORE THE LOOP? 
        #SOMEHOW UPDATE POINTS[] AND VECS[] IN-PLACE? USE @VIEWS? ==> TRY IN A *TOY EXAMPLE* FIRST!! 
        points[] = [Point2f(vars_scatter[i, j]...) for j in 1:num_points]
        vecs[] = [Point2f(vars_quiver[i, j]...) for j in 1:num_points]

        t_str = @sprintf("%5d", index[i])
        t_label[] = L"t = %$(t_str)"
    end

    video_bytes = read(filename)
    return video_bytes    
end

#######################################################################################################

#MAKIE VERSION OF display_video() 
function display_video(mp4_bytes::Vector{UInt8}; original_size::Tuple{Real, Real}, scale::Real = 1.0)

    original_width, original_height = original_size 

    width_scaled = Int(round(original_width * scale))
    height_scaled = Int(round(original_height * scale))

    b64 = base64encode(mp4_bytes)
    html = """
    <video width="$width_scaled" height="$height_scaled" controls autoplay loop muted>
        <source src="data:video/mp4;base64,$b64" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """
    display("text/html", html)
end


function save_mp4(bytes::Vector{UInt8}, filename::AbstractString)
    open(filename, "w") do io
        write(io, bytes)
    end
    # Example usage:
    # save_mp4(mp4_bytes, joinpath(pwd(), "blahh.mp4")
end


save_mp4 (generic function with 1 method)

In [28]:
#QUICK DEMO of simulate() and CREATING ANIMATIONS [DO NOT DELETE]
##################################################################

#1) SOLVE THE ODE 
N = 20 
sim = simulate(C = 1.0, ℓ = 1.0, α = 0.1, N = N, nframes = 201);  

#2) CONVERT THE SOLUTION TO A DATAFRAME (EACH COLUMN IS A VARIABLE)
x_cols = [Symbol("x", i) for i=1:N]
w_cols = [Symbol("w", i) for i=1:N]
var_names = vcat(x_cols, w_cols);
@time sol = ode_solution_to_df(sim.u, sim.t, var_dim = 1, var_names = var_names);


#3) SET SIZE OF MP4 AND GET THE TITLE STRING 
scale = 2.0
fig_size = scale .* (300, 250)
@unpack title_string, path_string = param_strings(sim.prob.p.constant);

title_string

Solve time: 3.196612 seconds

  0.196593 seconds (89.20 k allocations: 5.146 MiB, 97.55% compilation time)


L"$C = 1.00, \; ℓ = 1.00, \; α = 0.10, \; N = 20, \; \textrm{seed} = 1234 \;\;\;$"

In [17]:
latex_str = "C = 1.00, \\; \\ell = 1.00, \\; \\alpha = 0.10, \\; N = 20, \\; \\textrm{seed} = 1234"

L"%$(latex_str)"

L"$C = 1.00, \; \ell = 1.00, \; \alpha = 0.10, \; N = 20, \; \textrm{seed} = 1234$"

(3, 9)

In [29]:
#4) CREATE THE MP4 VIDEO 
@time mp4_bytes = animated_scatter_with_quiver(sol; index = sol.t, cols_scatter = x_cols, cols_quiver = w_cols, title = "",
                                               xlim = (-500, 500), ylim = (-500, 500), arrowlength = 60.0, size = fig_size, fps = 30,
                                               particle_diameter = 20);

# #5) DISPLAY THE VIDEO IN NOTEBOOK CELL 
# @time display_video(mp4_bytes; original_size = fig_size, scale = 0.95)

# #6) SAVE THE MP4 FILE  
# #@time save_mp4(mp4_bytes, joinpath(pwd(), "blahh.mp4"));

LoadError: UndefVarError: `Point2f` not defined

In [37]:
@show names(Main)  # to see if Point2f is defined

@show Point2f


names(Main) = [:Base, :Core, :ExternalCueBuffer, :ForceBuffer, :G, :K, :MT, :Main, :ModelCache, :ModelConstant, :ModelParameter, :MorseParameter, :N, :QuadratureData, :absolute_angular_momentum, :angular_momentum, :animated_scatter_with_quiver, :average_nearest_neighbors, :center_of_mass, :cluster_eccentricity, :compute_external_cues!, :compute_forces!, :create_mp4, :cue, :dU, :display_video, :fig_size, :gauss_legendre_quadrature_data, :group_polarization, :initialize_positions_and_polarities, :latex_str, :mean_interparticle_distance, :mt_fonts_dir, :normalize_rows!, :ode_solution_to_df, :output_dir, :param_strings, :param_strings_makie, :path_string, :save_mp4, :scale, :shreyasmodel2!, :silence_all_output, :sim, :simulate, :sol, :title_string, :var_names, :variable_names_shrey, :volex_magnitude, :w_cols, :x_cols]


LoadError: UndefVarError: `Point2f` not defined

In [36]:
using GeometryBasics

In [35]:
names(CairoMakie, all=true)


635-element Vector{Symbol}:
 Symbol("##CairoBackend#1")
 Symbol("##CairoScreen#2")
 Symbol("##CairoScreen#3")
 Symbol("##activate!#77")
 Symbol("##draw_mesh3D#40")
 Symbol("##meta#58")
 Symbol("#10#11")
 Symbol("#12#13")
 Symbol("#14#15")
 Symbol("#16#19")
 Symbol("#17#20")
 Symbol("#18#21")
 Symbol("#22#23")
 ⋮
 :ytickrotation
 :ytickrotation!
 :yticks!
 :zlabel!
 :zlims!
 :zoom!
 :zticklabels
 :ztickrange
 :ztickrotation
 :ztickrotation!
 :zticks!
 :zvalue